# NOC Ticket Classification System
RAG + prompt versioning + eval harness

In [ ]:
!pip install -q anthropic chromadb sentence-transformers

In [ ]:
import os
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'  # ← colle ta clé ici

## 1. Génération des 500 tickets

In [ ]:
import anthropic, json

client = anthropic.Anthropic()

TAXONOMY = ['fiber_cut', 'power_outage', 'router_misconfiguration', 'ddos_attack', 'planned_maintenance']

def generate_ticket(category: str, ticket_id: int) -> dict:
    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=300,
        messages=[{
            'role': 'user',
            'content': f'''Generate a realistic NOC (Network Operations Center) incident ticket
            for category: {category}.
            Include: timestamp, severity (P1-P4), affected equipment, symptoms,
            initial diagnosis. 2-3 sentences. Realistic telecom jargon.
            Return JSON only: {{"ticket_id": {ticket_id}, "text": "...", "true_label": "{category}"}}'''
        }]
    )
    return json.loads(response.content[0].text)

dataset = []
for category in TAXONOMY:
    print(f'Generating {category}...')
    for i in range(100):
        ticket = generate_ticket(category, len(dataset))
        dataset.append(ticket)
        if (i+1) % 10 == 0:
            print(f'  {i+1}/100')

with open('tickets_dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

print(f'Done: {len(dataset)} tickets generated')

## 2. Construction de la knowledge base (RAG)

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

RUNBOOKS = [
    {
        'id': 'rb_fiber_cut', 'category': 'fiber_cut',
        'content': '''Fiber cut resolution playbook:
        1. Identify affected segment via OTDR measurement
        2. Dispatch field team within 2h for P1, 4h for P2
        3. Activate backup path if available (MPLS FRR)
        4. Typical resolution time: 4-8h
        5. Escalate to regional NOC if >3 segments affected'''
    },
    {
        'id': 'rb_ddos', 'category': 'ddos_attack',
        'content': '''DDoS mitigation playbook:
        1. Confirm attack via traffic analysis (NetFlow)
        2. Activate scrubbing center if traffic >10Gbps
        3. Apply upstream BGP blackhole if customer requests
        4. Loop in Security SOC team immediately
        5. Document attack vector for post-mortem'''
    },
    {
        'id': 'rb_power', 'category': 'power_outage',
        'content': '''Power outage response playbook:
        1. Verify UPS status and estimated battery runtime
        2. Contact facility/data center team immediately
        3. Activate generator if outage >15min
        4. Monitor thermal thresholds on active equipment
        5. Initiate controlled shutdown if power not restored within SLA'''
    },
    {
        'id': 'rb_router_misconfig', 'category': 'router_misconfiguration',
        'content': '''Router misconfiguration playbook:
        1. Capture running config before any changes
        2. Compare with last known-good config (git/RANCID)
        3. Roll back to previous config if diff is clear
        4. Test BGP/OSPF adjacencies post-rollback
        5. Open change ticket and schedule post-mortem'''
    },
    {
        'id': 'rb_maintenance', 'category': 'planned_maintenance',
        'content': '''Planned maintenance playbook:
        1. Verify maintenance window is approved and notified
        2. Confirm backup paths are active before starting
        3. Execute change with rollback plan ready
        4. Monitor for 30min post-change
        5. Close maintenance ticket and notify stakeholders'''
    },
]

RESOLVED_INCIDENTS = [
    {
        'id': 'inc_001', 'category': 'router_misconfiguration',
        'content': "BGP session drop on PE-router Lyon-3 after IOS upgrade. Root cause: missing 'no auto-summary' command. Resolution: rolled back config, re-applied with fix. Duration: 47min."
    },
    {
        'id': 'inc_002', 'category': 'fiber_cut',
        'content': 'Fiber cut on Paris-Lyon backbone segment at km 342. Root cause: civil works by third party. Resolution: field team dispatched, temporary splice applied. Duration: 6h20min.'
    },
    {
        'id': 'inc_003', 'category': 'ddos_attack',
        'content': 'UDP flood targeting customer AS65001, peak 45Gbps. Root cause: compromised IoT botnet. Resolution: upstream blackhole + scrubbing center activation. Duration: 23min mitigation.'
    },
]

def build_collection():
    chroma_client = chromadb.Client()
    collection = chroma_client.create_collection('noc_knowledge_base')
    model = SentenceTransformer('all-MiniLM-L6-v2')
    all_docs = RUNBOOKS + RESOLVED_INCIDENTS
    embeddings = model.encode([d['content'] for d in all_docs]).tolist()
    collection.add(
        documents=[d['content'] for d in all_docs],
        embeddings=embeddings,
        ids=[d['id'] for d in all_docs],
        metadatas=[{'category': d['category']} for d in all_docs]
    )
    print(f'Knowledge base built: {len(all_docs)} documents indexed')
    return collection

collection = build_collection()

## 3. Classifier (v1 sans RAG, v2 avec RAG)

In [ ]:
import time
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

PROMPT_VERSIONS = {
    'v1': '''You are a NOC classification system.
Classify the following incident ticket into exactly one category: {taxonomy}.
Return JSON only: {{"classification": "category", "confidence": 0.0, "summary": "one sentence"}}

Ticket: {ticket}''',

    'v2': '''You are an expert NOC engineer at a major telecom operator.

RELEVANT CONTEXT FROM KNOWLEDGE BASE:
{rag_context}

Classify this incident ticket into exactly one of these categories: {taxonomy}
Rules:
- Choose the PRIMARY cause, not secondary effects
- If ambiguous between two categories, pick the most operationally actionable one
- Confidence < 0.7 means you're uncertain

Return JSON only:
{{"classification": "category", "confidence": 0.0, "summary": "one sentence max 20 words, action-oriented", "recommended_runbook": "runbook id if applicable"}}

Ticket: {ticket}'''
}

def retrieve_context(ticket_text, collection, n_results=2):
    query_embedding = embed_model.encode([ticket_text]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=n_results)
    return '\n'.join(f"- {doc[:200]}..." for doc in results['documents'][0])

def classify_ticket(ticket_text, collection, prompt_version='v2'):
    start = time.time()
    rag_context = retrieve_context(ticket_text, collection)
    prompt = PROMPT_VERSIONS[prompt_version].format(
        taxonomy=', '.join(TAXONOMY),
        rag_context=rag_context,
        ticket=ticket_text
    )
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=200,
        messages=[{'role': 'user', 'content': prompt}]
    )
    result = json.loads(response.content[0].text)
    result['latency_ms'] = round((time.time() - start) * 1000)
    result['prompt_version'] = prompt_version
    return result

print('Classifier ready')

## 4. Eval harness — compare v1 vs v2

In [ ]:
from collections import defaultdict

def run_evaluation(dataset_path, collection, prompt_version='v2', sample_size=500):
    with open(dataset_path) as f:
        dataset = json.load(f)[:sample_size]

    results = []
    confusion_matrix = defaultdict(lambda: defaultdict(int))

    for i, ticket in enumerate(dataset):
        prediction = classify_ticket(ticket['text'], collection, prompt_version)
        is_correct = prediction['classification'] == ticket['true_label']
        confusion_matrix[ticket['true_label']][prediction['classification']] += 1
        results.append({
            'ticket_id': ticket['ticket_id'],
            'true_label': ticket['true_label'],
            'predicted': prediction['classification'],
            'confidence': prediction['confidence'],
            'latency_ms': prediction['latency_ms'],
            'correct': is_correct
        })
        if (i+1) % 50 == 0:
            acc_so_far = sum(1 for r in results if r['correct']) / len(results)
            print(f'  [{prompt_version}] {i+1}/{len(dataset)} — acc so far: {acc_so_far:.1%}')

    total = len(results)
    correct = sum(1 for r in results if r['correct'])
    accuracy = correct / total
    avg_latency = sum(r['latency_ms'] for r in results) / total

    per_category = {}
    for category in set(r['true_label'] for r in results):
        cat_results = [r for r in results if r['true_label'] == category]
        per_category[category] = {
            'accuracy': sum(1 for r in cat_results if r['correct']) / len(cat_results),
            'count': len(cat_results)
        }

    report = {
        'prompt_version': prompt_version,
        'sample_size': total,
        'overall_accuracy': round(accuracy, 4),
        'acceptance_criteria_met': accuracy >= 0.85,
        'avg_latency_ms': round(avg_latency),
        'p95_latency_ms': sorted([r['latency_ms'] for r in results])[int(total * 0.95)],
        'per_category_accuracy': per_category,
        'confusion_matrix': {k: dict(v) for k, v in confusion_matrix.items()},
        'low_confidence_rate': sum(1 for r in results if r['confidence'] < 0.7) / total
    }
    return report

for version in ['v1', 'v2']:
    print(f'\nRunning eval — prompt {version}...')
    report = run_evaluation('tickets_dataset.json', collection, version)
    print(f'\n=== EVAL RESULTS — Prompt {version} ===')
    print(f"Accuracy:              {report['overall_accuracy']:.1%}")
    print(f"Criteria met (>85%):   {'YES' if report['acceptance_criteria_met'] else 'NO'}")
    print(f"Avg latency:           {report['avg_latency_ms']}ms")
    print(f"P95 latency:           {report['p95_latency_ms']}ms")
    print(f"Low confidence rate:   {report['low_confidence_rate']:.1%}")
    print(f"Per category:")
    for cat, stats in report['per_category_accuracy'].items():
        print(f"  {cat:30s} {stats['accuracy']:.1%}")
    with open(f"eval_report_{version}.json", 'w') as f:
        json.dump(report, f, indent=2)